# Preparing Lin et al. 2024 (mouse tracheal hillock cell) data for UCE
Universal Cell Embeddings trained on human cell atlas data
See https://github.com/snap-stanford/UCE  

Using `uce` conda environment. Should be visible on DCC.  May need to 
* `source ~/.bash_profile`
* `export PATH="$PATH:/usr/local/bin/:~/bin:/hpc/group/oliverlab/mambaforge/bin"`
* reload browser window running VS Code


#### Embedding a new dataset

To generate an embedding for a new single-cell RNA sequencing dataset in the AnnData format, use the `eval_single_anndata.py` script.

`python eval_single_anndata.py --adata_path {path_to_anndata} --dir {output_dir} --species {species} --model_loc {model_loc} --batch_size {batch_size}`

where  

* `adata_path`: a h5ad file. The .X slot of the file should be scRNA-seq counts. The .var_names slot should correspond to gene names, not ENSEMBLIDs.
* `dir`: the working directory in which intermediate and final output files will be saved to skip repeated processing of the same dataset.
* `species`: the species of the dataset you are embedding.
* `model_loc`: the location of the model weights .torch file.
* `batch_size`: the per GPU batch size. For the 33 layer model, on a 80GB GPU, you should use 25. For a 4 layer model on the same GPU, you can use 100.

In [1]:
import scanpy as sc
import os
from pathlib import Path
import sys
import numpy as np
import scipy.sparse as sp

def is_integer_matrix(adata):
    """Check if all elements in the sparse count matrix of an AnnData object are integers."""
    X = adata.X
    if sp.issparse(X):
        return np.all(np.equal(X.data, X.data.astype(int)))
    else:
        return np.all(np.equal(X, X.astype(int)))



/hpc/group/oliverlab/mambaforge/envs/uce/lib/python3.10/site-packages/anndata/utils.py:429: FutureWarning: Importing read_csv from `anndata` is deprecated. Import anndata.io.read_csv instead.
  warnings.warn(msg, FutureWarning)
/hpc/group/oliverlab/mambaforge/envs/uce/lib/python3.10/site-packages/anndata/utils.py:429: FutureWarning: Importing read_excel from `anndata` is deprecated. Import anndata.io.read_excel instead.
  warnings.warn(msg, FutureWarning)
/hpc/group/oliverlab/mambaforge/envs/uce/lib/python3.10/site-packages/anndata/utils.py:429: FutureWarning: Importing read_hdf from `anndata` is deprecated. Import anndata.io.read_hdf instead.
  warnings.warn(msg, FutureWarning)
/hpc/group/oliverlab/mambaforge/envs/uce/lib/python3.10/site-packages/anndata/utils.py:429: FutureWarning: Importing read_loom from `anndata` is deprecated. Import anndata.io.read_loom instead.
  warnings.warn(msg, FutureWarning)
/hpc/group/oliverlab/mambaforge/envs/uce/lib/python3.10/site-packages/anndata/util

### Load the data and confirm structure

In [2]:
DATADIR = Path("/hpc/group/oliverlab/lung_atlas")
file_path = DATADIR / "Lin_2024_hillock_QC.h5ad"
assert file_path.exists()

In [3]:
adata = sc.read_h5ad(file_path)
adata

AnnData object with n_obs × n_vars = 32156 × 26076
    obs: 'sample', 'n_genes_by_counts', 'total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'pct_counts_hb', 'log1p_total_counts', 'outlier', 'mt_outlier', 'n_genes', 'doublet_scores', 'predicted_doublets', 'doublet', 'doublet_score', 'predicted_doublet', 'leiden', 'epith', 'mouse_age'
    var: 'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm', 'highly_variable_nbatches'
    uns: 'hvg', 'leiden', 'leiden_colors', 'neighbors', 'pca', 'rank_genes_groups', 'sample_colors', 'scrublet', 'umap'
    obsm: 'X_pca', 'X_umap'
    varm: 'PCs'
    layers: 'counts', 'log1p_norm', 'norm'
    obsp: 'connectivities', 'distances'

In [4]:
assert is_integer_matrix(adata)

In [5]:
adata_path = file_path
output_dir = Path("~/work-drt42/UCE").expanduser()
species = "mouse"
model_loc = Path("/hpc/group/oliverlab/UCE/33l_8ep_1024t_1280.torch")
batch_size = 25

In [7]:
assert output_dir.exists()
assert model_loc.exists()

In [8]:
fxn_call = f"python eval_single_anndata.py --adata_path {adata_path} --dir {output_dir} --species {species} --model_loc {model_loc} --batch_size {batch_size}"
print(fxn_call)

python eval_single_anndata.py --adata_path /hpc/group/oliverlab/lung_atlas/Lin_2024_hillock_QC.h5ad --dir /hpc/home/drt42/work-drt42/UCE --species mouse --model_loc /hpc/group/oliverlab/UCE/33l_8ep_1024t_1280.torch --batch_size 25


### Try to execute from within this notebook